<a href="https://colab.research.google.com/github/GodesAleksandra/DI-Bootcamp/blob/main/Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [23]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [24]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.13.15
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [25]:
%%writefile server.py
#from mcp.server.fastmcp import FastMCP
from mcp.server.mcpserver import MCPServer

#mcp = FastMCP("Demo")
mcp = MCPServer("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    # TODO: return the sum
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    # TODO: return "Hello, {name}!"
    return f"Hello, {name}!"

if __name__ == "__main__":
    # TODO: start the server loop over STDIO
    mcp.run(transport="stdio")

Overwriting server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [26]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


"""
def extract_content(payload):
    #Best-effort to pull text from MCP responses.
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)
"""

def extract_content(payload):
    """Updated text extraction tailored for MCP v2 model specifications."""
    if hasattr(payload, "content"):
        # IF it is CallToolResult
        contents = payload.content
        if contents and hasattr(contents[0], "text"):
            return contents[0].text
    if hasattr(payload, "contents"):
        # IF it is ReadResourceResult
        contents = payload.contents
        if contents and hasattr(contents[0], "text"):
            return contents[0].text
    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # TODO: list resources and print their URIs
            resources_result = await session.list_resources()
            print("Resources:")
            for resource in resources_result.resources:
                print(f"  - {resource.uri}")

            # TODO: list tools and print their names
            tools_result = await session.list_tools()
            print("\nTools:")
            for tool in tools_result.tools:
                print(f"  - {tool.name}")

            # TODO: read greeting://hello and print the content
            print("\nReading resource 'greeting://hello':")
            resource_content = await session.read_resource("greeting://hello")
            print(f"  Result: {extract_content(resource_content)}")

            # TODO: call add with a=1, b=7 and print the result
            print("\nCalling tool 'add' with a=1, b=7:")
            tool_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(f"  Result: {extract_content(tool_result)}")


if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [27]:
# Run the client (spawns the server over STDIO)
!python client.py

Resources:

Tools:
  - add

Reading resource 'greeting://hello':
  Result: Hello, hello!

Calling tool 'add' with a=1, b=7:
  Result: 8


## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.